# HrudayAI — Milestone 2: Open-Source Symptom Extraction (Hindi & English)

This Google Colab CPU-compatible notebook loads Phase 1 transcription JSON (`phase1_voice/outputs/transcription_results.json`) and extracts structured symptoms (`canonical_name`, `original_term`, `duration`, `severity`, `negated`) using free, open-source local Python tools.

- **Languages:** Hindi (`hi`) and English (`en`)
- **External / Paid APIs:** None
- **Disease Prediction:** Not connected in this milestone

## 1. Environment & Dependency Setup (Colab CPU)

In [ ]:
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_ROOT = Path("/content/HrudayAI")
    if not REPO_ROOT.exists():
        token = None
        try:
            from google.colab import userdata  # type: ignore
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = os.environ.get("GITHUB_TOKEN")

        if token:
            subprocess.check_call([
                "git", "clone",
                f"https://{token}@github.com/itsakshatsinghh/HrudayAI.git",
                str(REPO_ROOT),
            ])
        else:
            raise RuntimeError(
                "Repository not found at /content/HrudayAI. "
                "Clone it first with `!git clone https://<TOKEN>@github.com/itsakshatsinghh/HrudayAI.git /content/HrudayAI`."
            )
    os.chdir(REPO_ROOT)
else:
    cwd = Path.cwd().resolve()
    REPO_ROOT = cwd.parent.parent if cwd.name == "notebooks" else cwd

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REQ_FILE = REPO_ROOT / "phase2_symptom_extraction" / "requirements.txt"
if IN_COLAB and REQ_FILE.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_FILE)])
    importlib.invalidate_caches()

print(f"Running in Colab : {IN_COLAB}")
print(f"Repo root        : {REPO_ROOT}")
print(f"rapidfuzz ready  : {importlib.util.find_spec('rapidfuzz') is not None}")

## 2. Load Phase 1 Transcription JSON (`phase1_voice/outputs/transcription_results.json`)

Reads `phase1_voice/outputs/transcription_results.json` if present in the runtime; otherwise allows uploading it in Colab or running with the verified Phase 1 Hindi & English reference outputs.

In [ ]:
PHASE1_OUTPUT_PATH = REPO_ROOT / "phase1_voice" / "outputs" / "transcription_results.json"

if PHASE1_OUTPUT_PATH.exists():
    phase1_payload = json.loads(PHASE1_OUTPUT_PATH.read_text(encoding="utf-8"))
    print(f"Loaded Phase 1 output from: {PHASE1_OUTPUT_PATH}")
else:
    print(f"{PHASE1_OUTPUT_PATH} not found locally; using verified Phase 1 sample payload.")
    phase1_payload = [
        {
            "detected_language": "hi",
            "text": "मुजे कल से बुखार है, सिर में दर्द हो रहा है, और तोडी कमजोरी भी महसुस हो रही है",
        },
        {
            "detected_language": "en",
            "text": "I have been experiencing fevers since yesterday along with a headache and some weaknesses.",
        },
    ]

print(json.dumps(phase1_payload, indent=2, ensure_ascii=False))

## 3. Run Structured Symptom Extraction on Phase 1 Output

In [ ]:
from phase2_symptom_extraction.src import extract_from_phase1_payload, extract_symptoms

extracted_results = extract_from_phase1_payload(phase1_payload)
print("=== Structured Symptom Extraction Output ===")
print(json.dumps(extracted_results, indent=2, ensure_ascii=False))

## 4. Test Negation, Duration, and Severity Edge Cases (Hindi & English)

In [ ]:
sample_cases = [
    ("hi", "मुझे दो दिन से तेज बुखार है, हल्की खांसी है, लेकिन उल्टी नहीं हो रही।"),
    ("en", "I have severe chest pain for 3 days, mild headache, and no fever or vomiting."),
]

for lang, text in sample_cases:
    out = extract_symptoms(text, language=lang)
    print(f"\n--- Case ({lang}) ---")
    print(json.dumps(out, indent=2, ensure_ascii=False))

## 5. Run Unit Tests Inside Colab

In [ ]:
import unittest

suite = unittest.defaultTestLoader.discover(str(REPO_ROOT / "phase2_symptom_extraction" / "tests"))
runner = unittest.TextTestRunner(verbosity=2)
test_result = runner.run(suite)
assert test_result.wasSuccessful(), "Phase 2 symptom extraction tests failed!"